# Many-against-Many sequence searching (MMseqs2)

This Jupyter Notebook is based on the guide of the Soeding Lab wiki. (https://github.com/soedinglab/mmseqs2/wiki#searching)

- Why? Clustering of data
- We also try Omega blasting 
- C++ nutzen? 
- 10000x schneller als BLAST (100 -> almost same sensitivity as BLAST)

### 1. Installation

DOWNLOAD mmseqs2 via binary or homebrew, conda or docker (not possible via apt) - https://github.com/soedinglab/MMseqs2
- i used brew on WSL with Linux https://linux.how2shout.com/install-brew-on-wsl-windows-subsystem-for-linux/
-  (Ubunutu WSL): brew install mmseqs2

### 2. Getting started

TODO: Run search for sequences matched -> query database against target database https://github.com/soedinglab/mmseqs2/wiki#getting-started
- DOWNLOAD Test databases in example folder also for clustering

#### 2.1 Import packages

In [ ]:
# load standard packages
import pandas as pd
import os
import numpy as np
import seaborn as sns
from tabulate import tabulate # package install: sudo apt install python3-tabulate

In [ ]:
# IMPORTANT subprocess needed because mmseqs2 is a command-line tool (no Python pachage)
# to run external commands 
import subprocess

# Options of subprocess
# subprocess.run(
#    ["command", "arg1", "arg2"],   # list of commands
#     check=True,                   # Error if command fails
#     capture_output=True,          # Capture stdout/stderr instead of printing
#     text=True                     # Decode bytes to string
#
# )

# First experiment
# subprocess.run(["mmseqs", "createdb", "input.fasta", "inputDB"], check=True)
subprocess.run(["mmseqs", "createdb", "../data/rcsb_pdb_7XP6.fasta", "../data/DB.fasta"], check=True)


#### MMseqs2 moduls

In [3]:
# Input into terminal!!! 
# Structure to call MMseqs2 modul; module, arg are parameter and options for behavior or parameter settings 
!mmseqs module input_db output_db args [options]

# For HELP a complete list of modules
!mmseqs -h

# useful are for example the modules Eary-Search/Cluster/Linclust

MMseqs2 (Many against Many sequence searching) is an open-source software suite for very fast, 
parallelized protein sequence searches and clustering of huge protein sequence data sets.

Please cite: M. Steinegger and J. Soding. MMseqs2 enables sensitive protein sequence searching for the analysis of massive data sets. Nature Biotechnology, doi:10.1038/nbt.3988 (2017).

MMseqs2 Version: 17-b804f
© Martin Steinegger (martin.steinegger@snu.ac.kr)

usage: mmseqs <command> [<args>]

Easy workflows for plain text input/output
  easy-search       	Sensitive homology search
  easy-linsearch    	Fast, less sensitive homology search
  easy-cluster      	Slower, sensitive clustering
  easy-linclust     	Fast linear time cluster, less sensitive clustering
  easy-taxonomy     	Taxonomic classification
  easy-rbh          	Find reciprocal best hit

Main workflows for database input/output
  search            	Sensitive homology search
  linsearch         	Fast, less sensitive homology search
  map 

### Download and preperation of data

Two possible approche
- Finding and setting up databases on our own
- OR use provided databases 

Luckly they provide ready database for homology searches on protein, nucleotide, profile and taxonomy

In [ ]:
# List of all available data bases
!mmseqs databases -h

usage: mmseqs databases <name> <o:sequenceDB> <tmpDir> [options]
 By Milot Mirdita <milot@mirdita.de>

  Name                	Type      	Taxonomy	Url                                                           
- UniRef100           	Aminoacid 	     yes	https://www.uniprot.org/help/uniref
  The UniProt Reference Clusters provide clustered sets of sequences from the UniProt Knowledgebase.
  Cite: Suzek et al: UniRef: comprehensive and non-redundant UniProt reference clusters. Bioinformatics 23(10), 1282–1288 (2007)

- UniRef90            	Aminoacid 	     yes	https://www.uniprot.org/help/uniref
  The UniProt Reference Clusters provide clustered sets of sequences from the UniProt Knowledgebase.
  Cite: Suzek et al: UniRef: comprehensive and non-redundant UniProt reference clusters. Bioinformatics 23(10), 1282–1288 (2007)

- UniRef50            	Aminoacid 	     yes	https://www.uniprot.org/help/uniref
  The UniProt Reference Clusters provide clustered sets of sequences from the UniProt Knowle

#### PDB Download

In [ ]:
# Avoid "file not found" by using mkdir -p <path> 
!mkdir -p ../db ../tmp

# Downloading pdb files in ../group04-team03/blasting/mmseqs2/db and tmp for temporary data into ../group04-team03/blasting/blasting/mmseqs2/tmp
# TODO -> only download if pdb files do not exist in db; maybe using if and else
!mmseqs databases PDB ../db/pdb ../tmp

databases PDB ../db/pdb ../tmp 

MMseqs Version:              	17-b804f
Tsv                          	false
Force restart with latest tmp	false
Remove temporary files       	false
Compressed                   	0
Threads                      	16
Verbosity                    	3

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 55.1M  100 55.1M    0     0  9310k      0  0:00:06  0:00:06 --:--:-- 11.4M
createdb ../tmp/16973763654333910291/pdb_seqres.txt.gz ../db/pdb --compressed 0 -v 3 

Converting sequences
[1007678] 1s 831ms
Time for merging to pdb_h: 0h 0m 0s 133ms
Time for merging to pdb: 0h 0m 0s 651ms
Database type: Aminoacid
Time for processing: 0h 0m 3s 420ms


#### Query DB

In [34]:
# EVEscape https://evescape.org/variantsofconcern
# FASTA with protein of Omicron BF.7 saved to ../blasting/mmseqs2/data/query.fasta
import requests
from pathlib import Path

# (PDB ID + chain)  
entries = [
    ("7bnn", "A"),
    ("7bnn", "B"),
    ("7bnn", "C"),
    ("7cab", "A"),
    ("7cab", "B"),
    ("7cab", "C"),
    ("6vxc", "A"),
]

# Goal path
output_dir = Path("../data")
output_dir.mkdir(exist_ok=True)
query_fasta = output_dir / "queryOmicron.fasta"

# Creating fasta
with open(query_fasta, "w") as out_f:
    for pdb_id, chain_id in entries:
        url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display?chainId={chain_id}"
        print(f"Fetching {pdb_id}_{chain_id}...")
        resp = requests.get(url)

        if resp.status_code == 200 and resp.text.strip():
            out_f.write(resp.text.strip() + "\n")
        else:
            print(f"Failed to fetch {pdb_id}_{chain_id} (Status {resp.status_code})")

print(f"\n FASTA saved to: {query_fasta}")



Fetching 7bnn_A...
Fetching 7bnn_B...
Fetching 7bnn_C...
Fetching 7cab_A...
Fetching 7cab_B...
Fetching 7cab_C...
Fetching 6vxc_A...

 FASTA saved to: ../data/queryOmicron.fasta


In [37]:
# Create query DB
!mkdir -p ../data/queryOmicron.fasta
!mmseqs createdb ../data/queryOmicron.fasta ../db/query/omicron


mkdir: cannot create directory ‘../data/queryOmicron.fasta’: File exists
createdb ../data/queryOmicron.fasta ../db/query/omicron 

MMseqs Version:       	17-b804f
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed            	0
Verbosity             	3

Converting sequences

Time for merging to omicron_h: 0h 0m 0s 0ms
Time for merging to omicron: 0h 0m 0s 0ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 13ms


### Searching

In [43]:
# searching omicron against pdb; output is a BLAST tabel via .m8
!mmseqs search ../db/query/omicron ../db/pdb ../results/omicron_vs_pdb.m8 ../tmp -s 7.5

search ../db/query/omicron ../db/pdb ../results/omicron_vs_pdb.m8 ../tmp -s 7.5 

MMseqs Version:                        	17-b804f
Substitution matrix                    	aa:blosum62.out,nucl:nucleotide.out
Add backtrace                          	false
Alignment mode                         	2
Alignment mode                         	0
Allow wrapped scoring                  	false
E-value threshold                      	0.001
Seq. id. threshold                     	0
Min alignment length                   	0
Seq. id. mode                          	0
Alternative alignments                 	0
Coverage threshold                     	0
Coverage mode                          	0
Max sequence length                    	65535
Compositional bias                     	1
Compositional bias                     	1
Max reject                             	2147483647
Max accept                             	2147483647
Include identical seq. id.             	false
Preload mode                           	0

In [54]:
# Analyse results with pandas
cols = [
    "query", "target", "pident", "evalue",
    "bitscore", "alnlen", "qstart", "qend", "tstart", "tend"
]

df = pd.read_csv("../results/omicron_vs_pdb.m8.1", sep="\t", names=cols)
df.head()

,query,target,pident,evalue,bitscore,alnlen,qstart,qend,tstart,tend
0,17949.0,1680.0,0.984,0.0,0.0,808.0,809.0,0.0,808.0,809.0
1,49442.0,1680.0,0.984,0.0,0.0,808.0,809.0,0.0,808.0,809.0
2,584807.0,1680.0,0.984,0.0,0.0,808.0,809.0,0.0,808.0,809.0
3,616299.0,1680.0,0.984,0.0,0.0,808.0,809.0,0.0,808.0,809.0
4,647791.0,1680.0,0.984,0.0,0.0,808.0,809.0,0.0,808.0,809.0
